In [2]:
class GraphAdjList:
    def __init__(self):
        self.adj_list = {}  
    
    def size(self) -> int:
        return len(self.adj_list)
    
    def add_edge(self, vet1, vet2, weight: int = 1):
        if vet1 not in self.adj_list or vet2 not in self.adj_list or vet1 == vet2:
            raise ValueError(f"顶点不存在或相同顶点: {vet1}, {vet2}")
        
        self.adj_list[vet1][vet2] = weight
        self.adj_list[vet2][vet1] = weight
    
    def remove_edge(self, vet1, vet2):
        if vet1 not in self.adj_list or vet2 not in self.adj_list or vet1 == vet2:
            raise ValueError(f"顶点不存在或相同顶点: {vet1}, {vet2}")
        
        self.adj_list[vet1].pop(vet2, None)
        self.adj_list[vet2].pop(vet1, None)
    
    def add_vertex(self, vet):
        if vet in self.adj_list:
            return
        self.adj_list[vet] = {}
    
    def remove_vertex(self, vet):
        if vet not in self.adj_list:
            raise ValueError(f"顶点不存在: {vet}")
        
        for vertex in self.adj_list:
            if vet in self.adj_list[vertex]:
                self.adj_list[vertex].pop(vet)

        self.adj_list.pop(vet)
    
    def print(self):
        print("邻接表（有权图）=")
        for vertex in self.adj_list:
            edges = [f"{v}({w})" for v, w in self.adj_list[vertex].items()]
            print(f"{vertex}: {edges},")
    
    def get_vertices(self):
        return list(self.adj_list.keys())
    
    def get_neighbors(self, vet):
        if vet not in self.adj_list:
            raise ValueError(f"顶点不存在: {vet}")
        return [(neighbor, weight) for neighbor, weight in self.adj_list[vet].items()]
    
    def get_edge_weight(self, vet1, vet2) -> int:
        if vet1 not in self.adj_list or vet2 not in self.adj_list[vet1]:
            raise ValueError(f"边不存在: {vet1}-{vet2}")
        return self.adj_list[vet1][vet2]
    
    def has_edge(self, vet1, vet2) -> bool:
        return vet1 in self.adj_list and vet2 in self.adj_list[vet1]

In [7]:
g = GraphAdjList()

for i in range(9):
    g.add_vertex(f"V{i}")

edges = [
    ("V0", "V1", 3), ("V0", "V5", 4),
    ("V1", "V2", 8), ("V1", "V8", 5), ("V1", "V6", 6),
    ("V5", "V6", 7), ("V5", "V4", 18),
    ("V2", "V8", 2), ("V2", "V3", 12),
    ("V8", "V3", 11),
    ("V6", "V3", 14), ("V6", "V7", 9),
    ("V3", "V7", 6), ("V3", "V4", 10),
    ("V7", "V4", 1)
]

for u, v, w in edges:
    g.add_edge(u, v, w)

g.print()
        

邻接表（有权图）=
V0: ['V1(3)', 'V5(4)'],
V1: ['V0(3)', 'V2(8)', 'V8(5)', 'V6(6)'],
V2: ['V1(8)', 'V8(2)', 'V3(12)'],
V3: ['V2(12)', 'V8(11)', 'V6(14)', 'V7(6)', 'V4(10)'],
V4: ['V5(18)', 'V3(10)', 'V7(1)'],
V5: ['V0(4)', 'V6(7)', 'V4(18)'],
V6: ['V1(6)', 'V5(7)', 'V3(14)', 'V7(9)'],
V7: ['V6(9)', 'V3(6)', 'V4(1)'],
V8: ['V1(5)', 'V2(2)', 'V3(11)'],


In [9]:
import heapq

class LazyPrimMST:
    def __init__(self, graph):
        self.graph = graph
        self.visited = set()
        self.pq = []  
        self.mst_edges = [] 
        self.total_weight = 0
    
    def _visit(self, vertex):
        self.visited.add(vertex)
        
        for neighbor, weight in self.graph.get_neighbors(vertex):
            if neighbor not in self.visited:
                heapq.heappush(self.pq, (weight, vertex, neighbor))
    
    def find_mst(self, start_vertex=None):
        # 选择起始顶点
        if start_vertex is None:
            vertices = self.graph.get_vertices()
            if not vertices:
                raise ValueError("图为空")
            start_vertex = vertices[0]
        
        # 初始化
        self.visited.clear()
        self.pq.clear()
        self.mst_edges.clear()
        self.total_weight = 0
        
        # 从起始顶点开始
        self._visit(start_vertex)
        
        # 主循环
        num_vertices = self.graph.size()
        while self.pq and len(self.mst_edges) < num_vertices - 1:
            # 弹出最小权重的边
            weight, u, v = heapq.heappop(self.pq)
            
            # 懒惰检查：如果两个端点都已访问，跳过
            if u in self.visited and v in self.visited:
                continue
            
            # 将边加入MST
            self.mst_edges.append((u, v, weight))
            self.total_weight += weight
            
            # 访问未访问的顶点
            new_vertex = v if v not in self.visited else u
            self._visit(new_vertex)
        
        # 检查是否找到MST
        if len(self.mst_edges) != num_vertices - 1:
            raise ValueError(f"图不连通，无法生成最小生成树。"
                           f"找到{len(self.mst_edges)}条边，需要{num_vertices - 1}条边")
        
        return self.mst_edges, self.total_weight
    
    def print_mst(self):
        print("\n最小生成树结果：")
        print("=" * 50)
        for u, v, w in self.mst_edges:
            print(f"  {u} -- {v} \t权重: {w}")
        print(f"\n最小生成树总权重: {self.total_weight}")
        print("=" * 50)

In [10]:
lazy_prim = LazyPrimMST(g)

print("\n从 V0 开始查找最小生成树：")
mst_edges, total_weight = lazy_prim.find_mst("V0")
lazy_prim.print_mst()


从 V0 开始查找最小生成树：

最小生成树结果：
  V0 -- V1 	权重: 3
  V0 -- V5 	权重: 4
  V1 -- V8 	权重: 5
  V8 -- V2 	权重: 2
  V1 -- V6 	权重: 6
  V6 -- V7 	权重: 9
  V7 -- V4 	权重: 1
  V7 -- V3 	权重: 6

最小生成树总权重: 36
